# NLP Customer Service Ticket Routing (Infosys Topaz brief, Task 2)

**Business problem:** Helpdesks lose time when tickets land in the wrong queue. We build a multiclass text classifier that routes each ticket to the right team and flags low-confidence tickets for human triage.

**Type:** multiclass text classification  |  **Audience:** internal helpdesk / service-desk managers  |  **Outcome:** predicted queue + confidence, deployed as a Streamlit app

**Target:** `Topic_group` (8 queues)  |  **Metrics:** accuracy, macro F1 (queues are imbalanced), per-class precision/recall

**Plan:** load → clean → EDA → features (TF-IDF) → compare models → evaluate and error analysis → save model → insights

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from preprocess import clean_text

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Load data
Dataset: IT service tickets (`Document` = ticket text, `Topic_group` = target queue). The text is already partly pre-processed (lower-cased, stop words removed).

In [ ]:
df = pd.read_csv(ROOT / "data" / "tickets.csv")
print(df.shape)
df.head()

## 3. Data cleaning

In [ ]:
print("Missing values:\n", df.isna().sum(), sep="")
print("Duplicate rows:", df.duplicated().sum())
print("\nColumn types:\n", df.dtypes, sep="")

In [ ]:
# Decisions: drop duplicates / nulls, cast text to str, normalise text with the same function the app uses
df = df.drop_duplicates().dropna().reset_index(drop=True)
df["Document"] = df["Document"].astype(str)
df["text"] = df["Document"].map(clean_text)       # lower, remove URLs/emails/digits/punctuation
df = df[df["text"].str.len() > 0].reset_index(drop=True)
df["n_words"] = df["text"].str.split().str.len()
print("Rows after cleaning:", len(df))
df[["Document", "text"]].head(3)

## 4. Exploratory data analysis

In [ ]:
# Univariate: class balance and text length
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
order = df["Topic_group"].value_counts().index
sns.countplot(data=df, y="Topic_group", order=order, ax=ax[0])
ax[0].set_title("Tickets per queue")
sns.histplot(df["n_words"].clip(upper=300), bins=50, ax=ax[1])
ax[1].set_title("Ticket length in words (clipped at 300)")
plt.tight_layout(); plt.show()
print((df["Topic_group"].value_counts(normalize=True) * 100).round(1))
print(df["n_words"].describe().round(1))

In [ ]:
# Bivariate: length vs queue
plt.figure(figsize=(10, 4.5))
sns.boxplot(data=df, y="Topic_group", x=df["n_words"].clip(upper=200), order=order, showfliers=False)
plt.title("Ticket length by queue"); plt.xlabel("words"); plt.show()

In [ ]:
# Most distinctive words per queue (mean TF-IDF weight)
eda_vec = TfidfVectorizer(min_df=5, max_df=0.5, sublinear_tf=True)
M = eda_vec.fit_transform(df["text"])
terms = np.array(eda_vec.get_feature_names_out())
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, cls in zip(axes.ravel(), order):
    idx = np.where(df["Topic_group"].values == cls)[0]
    w = np.asarray(M[idx].mean(axis=0)).ravel()
    top = w.argsort()[-10:]
    ax.barh(terms[top], w[top]); ax.set_title(cls)
plt.tight_layout(); plt.show()

In [ ]:
# Multivariate: how similar are the queues in vocabulary? (cosine similarity of mean TF-IDF vectors)
centroids = np.vstack([np.asarray(M[np.where(df["Topic_group"].values == c)[0]].mean(axis=0)).ravel() for c in order])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
sim = pd.DataFrame(centroids @ centroids.T, index=order, columns=order)
plt.figure(figsize=(8, 6)); sns.heatmap(sim, annot=True, fmt=".2f", cmap="viridis"); plt.title("Queue vocabulary similarity"); plt.show()

**EDA takeaways**
- Classes are imbalanced (Hardware is the largest queue, Internal Project / Administrative rights the smallest), so macro F1 and `class_weight="balanced"` matter.
- Most tickets are short (median about 26 words) with a long tail, so TF-IDF with n-grams is a sensible, fast representation.
- Each queue has clearly distinctive vocabulary; queue pairs with high similarity in the heatmap are where confusion is expected, and the confusion matrix below checks this.

## 5. Feature engineering
Text is converted to **TF-IDF unigrams and bigrams** (`min_df=3` drops rare typos, `max_df=0.9` drops near-universal words, `sublinear_tf` dampens repeated words). Bigrams capture phrases such as "reset password" or "disk space".

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["Topic_group"], test_size=0.2, stratify=df["Topic_group"], random_state=RANDOM_STATE)
print(len(X_train), "train /", len(X_test), "test")

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True)

vec = tfidf().fit(X_train)
print("Vocabulary size:", len(vec.vocabulary_))

## 6. Model building and comparison

In [ ]:
candidates = {
    "Complement Naive Bayes": ComplementNB(),
    "Linear SVM": LinearSVC(C=0.5, class_weight="balanced", random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(C=5, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
}
fitted, rows = {}, []
for name, clf in candidates.items():
    pipe = Pipeline([("tfidf", tfidf()), ("clf", clf)]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                 "Macro F1": f1_score(y_test, pred, average="macro"),
                 "Weighted F1": f1_score(y_test, pred, average="weighted")})
comparison = pd.DataFrame(rows).set_index("Model").round(4)
comparison

In [ ]:
# 3-fold cross-validation on the training set for the two strongest models (checks the gap is not luck)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
for name in ["Linear SVM", "Logistic Regression"]:
    s = cross_val_score(Pipeline([("tfidf", tfidf()), ("clf", candidates[name])]), X_train, y_train, cv=cv, scoring="f1_macro")
    print(f"{name}: CV macro F1 = {s.mean():.4f} +/- {s.std():.4f}")

**Model selection:** Logistic Regression and Linear SVM are practically tied and clearly beat Naive Bayes. **Logistic Regression is the final model** because it outputs class probabilities. The app uses these as a confidence score to decide between auto-routing and manual triage, which a plain SVM cannot do without extra calibration.

## 7. Evaluation

In [ ]:
final = fitted["Logistic Regression"]
pred = final.predict(X_test)
print(classification_report(y_test, pred, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45, cmap="Blues",
                                        normalize="true", values_format=".2f", ax=ax, colorbar=False)
ax.set_title("Normalised confusion matrix (test set)"); plt.tight_layout(); plt.show()

In [ ]:
# What the model learned: strongest words per queue
clf = final.named_steps["clf"]; names = np.array(final.named_steps["tfidf"].get_feature_names_out())
for i, cls in enumerate(clf.classes_):
    print(f"{cls:22s}", ", ".join(names[clf.coef_[i].argsort()[-8:][::-1]]))

In [ ]:
# Confidence vs accuracy: the basis for the human-triage threshold used in the app
proba = final.predict_proba(X_test)
conf = proba.max(axis=1); correct = (final.classes_[proba.argmax(axis=1)] == y_test.values)
rows = []
for t in [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    rows.append({"threshold": t, "auto-routed share": m.mean(), "accuracy of auto-routed": correct[m].mean()})
pd.DataFrame(rows).round(3)

In [ ]:
# Error analysis: look at confidently wrong tickets
err = pd.DataFrame({"text": X_test.values, "true": y_test.values, "pred": final.classes_[proba.argmax(axis=1)], "conf": conf.round(2)})
err = err[err["true"] != err["pred"]].sort_values("conf", ascending=False)
err.head(8).assign(text=lambda d: d["text"].str[:110])

## 8. Save the model

In [ ]:
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump(final, ROOT / "models" / "ticket_classifier.joblib", compress=3)
print("saved")

## 9. Insights and next steps

- The model routes about 86 to 87% of tickets correctly (macro F1 about 0.87) with a fast, explainable TF-IDF + Logistic Regression pipeline.
- Raising the confidence threshold trades coverage for accuracy. At a 0.7 threshold about 72% of tickets are auto-routed at roughly 95% accuracy, and the rest go to a human (at 0.9: about 51% auto-routed at roughly 98.5%). Choose the threshold from the cost of a misrouted ticket vs a manual review.
- Most remaining errors are between the three biggest queues, mainly HR Support vs Hardware and Hardware vs Miscellaneous, which suggests overlapping or noisy labels; label cleaning would help more than a bigger model.
- **Next:** try BERT / sentence-transformer embeddings (the brief's advanced option), tune hyper-parameters, and add active learning so human-triaged tickets are fed back as training data.